# Reproducing Table 3: ImageNet Models on CounterAnimal

This notebook reproduces the ImageNet model results from Table 3 in the paper:
**"A Sober Look at the Robustness of CLIPs to Spurious Features"**

## Features:
- **15 Models**: All ImageNet models from Table 3


In [ ]:
# Install required packages
!pip install torch torchvision tqdm pandas pillow

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image, ImageFile
import os
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Allow loading truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = 'path_to_dataset'


# Verify dataset exists
if os.path.exists(DATASET_PATH):
    num_classes = len([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d)) and d[0].isdigit()])
    print(f"Dataset found at {DATASET_PATH}")
    print(f"Found {num_classes} animal classes")
else:
    print(f"Dataset not found at {DATASET_PATH}")

## Model Configuration

Define all ImageNet models to evaluate (matching Table 3 in the paper)

In [ ]:
# Define all ImageNet models to evaluate
MODEL_CONFIGS = {
    # AlexNet
    'AlexNet': {
        'model_fn': lambda: models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    # VGG models
    'VGG-11': {
        'model_fn': lambda: models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'VGG-13': {
        'model_fn': lambda: models.vgg13(weights=models.VGG13_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'VGG-19': {
        'model_fn': lambda: models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    # ResNet models
    'RN-18': {
        'model_fn': lambda: models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'RN-34': {
        'model_fn': lambda: models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'RN-50': {
        'model_fn': lambda: models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'RN-101': {
        'model_fn': lambda: models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    # Vision Transformer models
    'ViT-B/16': {
        'model_fn': lambda: models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'ViT-B/32': {
        'model_fn': lambda: models.vit_b_32(weights=models.ViT_B_32_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'ViT-L/16': {
        'model_fn': lambda: models.vit_l_16(weights=models.ViT_L_16_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'ViT-L/32': {
        'model_fn': lambda: models.vit_l_32(weights=models.ViT_L_32_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    # ConvNeXt models
    'ConvNext-S': {
        'model_fn': lambda: models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'ConvNext-B': {
        'model_fn': lambda: models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
    'ConvNext-L': {
        'model_fn': lambda: models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1),
        'input_size': 224,
    },
}

print(f"Configured {len(MODEL_CONFIGS)} models for evaluation")
print("Models:", list(MODEL_CONFIGS.keys()))

BATCH_SIZE = 64

## Evaluation Functions

In [ ]:
def get_transform(input_size=224):
    """
    Get ImageNet-standard preprocessing transforms.
    """
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])


def process_images_batch(image_paths, model, transform, gt_label, device='cuda', batch_size=32):
    """
    Process multiple images in batches for faster inference.

    Args:
        image_paths: List of image file paths
        model: PyTorch model
        transform: Image preprocessing transform
        gt_label: Ground truth label
        device: Device to run on
        batch_size: Number of images to process at once

    Returns:
        (num_correct, num_total)
    """
    correct = 0
    total = 0

    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        batch_images = []

        for img_path in batch_paths:
            try:
                img = Image.open(img_path).convert('RGB')
                img_tensor = transform(img)
                batch_images.append(img_tensor)
            except Exception as e:
                continue

        if not batch_images:
            continue

        batch_tensor = torch.stack(batch_images).to(device)

        with torch.no_grad():
            outputs = model(batch_tensor)
            preds = outputs.argmax(1)

        # Count correct predictions
        correct += (preds == gt_label).sum().item()
        total += len(batch_images)

    return correct, total


def evaluate_model_on_class(model, class_path, transform, device='cuda', batch_size=32):
    """
    Evaluate a model on a single animal class directory using batch processing.

    Args:
        model: PyTorch model
        class_path: Path to class directory (e.g., "0 ostrich")
        transform: Image preprocessing transform
        device: Device to run inference on
        batch_size: Batch size for inference

    Returns:
        dict with easy_accuracy, hard_accuracy, and drop
    """
    # Extract ground truth label from directory name
    class_name = os.path.basename(class_path)
    gt_label = int(class_name.split(' ')[0])

    # Find easy and hard subdirectories
    subdirs = os.listdir(class_path)
    easy_dirs = [d for d in subdirs if d.startswith('common-')]
    hard_dirs = [d for d in subdirs if d.startswith('counter-')]

    if not easy_dirs or not hard_dirs:
        return None

    # Collect all easy image paths
    easy_image_paths = []
    for easy_dir in easy_dirs:
        easy_path = os.path.join(class_path, easy_dir)
        if not os.path.isdir(easy_path):
            continue

        image_files = [f for f in os.listdir(easy_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        easy_image_paths.extend([os.path.join(easy_path, f) for f in image_files])

    # Collect all hard image paths
    hard_image_paths = []
    for hard_dir in hard_dirs:
        hard_path = os.path.join(class_path, hard_dir)
        if not os.path.isdir(hard_path):
            continue

        image_files = [f for f in os.listdir(hard_path)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        hard_image_paths.extend([os.path.join(hard_path, f) for f in image_files])

    if not easy_image_paths or not hard_image_paths:
        return None

    # Process easy images in batches
    easy_correct, easy_total = process_images_batch(
        easy_image_paths, model, transform, gt_label, device, batch_size
    )

    # Process hard images in batches
    hard_correct, hard_total = process_images_batch(
        hard_image_paths, model, transform, gt_label, device, batch_size
    )

    if easy_total == 0 or hard_total == 0:
        return None

    easy_acc = 100.0 * easy_correct / easy_total
    hard_acc = 100.0 * hard_correct / hard_total
    drop = easy_acc - hard_acc

    return {
        'easy_acc': easy_acc,
        'hard_acc': hard_acc,
        'drop': drop,
        'easy_correct': easy_correct,
        'easy_total': easy_total,
        'hard_correct': hard_correct,
        'hard_total': hard_total,
    }


def evaluate_model(model_name, dataset_path, device='cuda', batch_size=32):
    """
    Evaluate a single model on the entire CounterAnimal dataset.
    """
    import time
    start_time = time.time()

    print(f"\n{'='*60}")
    print(f"Evaluating {model_name} (batch_size={batch_size})")
    print(f"{'='*60}")

    # Load model
    config = MODEL_CONFIGS[model_name]
    model = config['model_fn']()
    model = model.to(device)
    model.eval()

    # Get transform
    transform = get_transform(config['input_size'])

    # Find all class directories
    class_dirs = [d for d in os.listdir(dataset_path)
                  if os.path.isdir(os.path.join(dataset_path, d))
                  and d[0].isdigit()]
    class_dirs = sorted(class_dirs, key=lambda x: int(x.split(' ')[0]))

    print(f"Found {len(class_dirs)} classes")

    # Evaluate on each class
    results = []
    for class_dir in tqdm(class_dirs, desc=f"Evaluating {model_name}"):
        class_path = os.path.join(dataset_path, class_dir)
        result = evaluate_model_on_class(model, class_path, transform, device, batch_size)

        if result is not None:
            result['class'] = class_dir
            results.append(result)

    # Calculate overall statistics
    if not results:
        print(f"No valid results for {model_name}")
        return None

    easy_acc_avg = sum(r['easy_acc'] for r in results) / len(results)
    hard_acc_avg = sum(r['hard_acc'] for r in results) / len(results)
    drop_avg = easy_acc_avg - hard_acc_avg

    elapsed_time = time.time() - start_time

    overall_results = {
        'model': model_name,
        'easy': round(easy_acc_avg, 2),
        'hard': round(hard_acc_avg, 2),
        'drop': round(drop_avg, 2),
        'num_classes': len(results),
        'time_seconds': round(elapsed_time, 2),
        'per_class_results': results
    }

    print(f"\nResults for {model_name}:")
    print(f"  Easy: {easy_acc_avg:.2f}%")
    print(f"  Hard: {hard_acc_avg:.2f}%")
    print(f"  Drop: {drop_avg:.2f}%")
    print(f"  Time: {elapsed_time:.1f}s ({elapsed_time/60:.1f} min)")

    # Clean up
    del model
    torch.cuda.empty_cache()

    return overall_results

## Run Evaluation

Choose which models to evaluate:
- Set `models_to_eval = None` to evaluate all models
- Or specify a list, e.g., `models_to_eval = ['RN-50', 'ViT-B/16']`

In [ ]:
# Configure which models to evaluate
models_to_eval = None  # Set to None to evaluate all, or specify list

if models_to_eval is None:
    models_to_eval = list(MODEL_CONFIGS.keys())

print(f"Will evaluate {len(models_to_eval)} models:")
for model_name in models_to_eval:
    print(f"  - {model_name}")

In [ ]:
# Run evaluation on all selected models
all_results = []
output_dir = '/content/imagenet_results'
os.makedirs(output_dir, exist_ok=True)

for model_name in models_to_eval:
    try:
        result = evaluate_model(model_name, DATASET_PATH, device, batch_size=BATCH_SIZE)
        if result is not None:
            all_results.append(result)

            # Save individual result
            result_file = os.path.join(output_dir, f"{model_name.replace('/', '_')}_results.json")
            with open(result_file, 'w') as f:
                json.dump(result, f, indent=2)
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n✓ Completed evaluation of {len(all_results)} models")

## Display Results

In [ ]:
# Create summary table
if all_results:
    summary_df = pd.DataFrame([
        {
            'backbone': r['model'],
            'easy': r['easy'],
            'hard': r['hard'],
            'drop': r['drop'],
        }
        for r in all_results
    ])

    # Save summary
    summary_csv = os.path.join(output_dir, 'summary_results.csv')
    summary_df.to_csv(summary_csv, index=False)

else:
    print("No results to display")